# 项目 1｜用朴素贝叶斯推荐商品

一位顾客**只买了游戏机**。根据手头的消费记录，给这位顾客推荐**啤酒**还是**口红**？

第一部分用 Python 列表从头实现朴素贝叶斯，从数出匹配条数到算出类别概率。第二部分用 scikit-learn 准备训练数据、训练模型并预测，再与第一部分的结果核对。请从上到下运行代码单元。

## 第一部分｜用 Python 实现朴素贝叶斯

### 1. 读懂消费记录

每行是一条消费记录，按工具、杂志、电影票、游戏机的顺序排列。`1` 表示买过，`0` 表示没买过。

记录按已知的推荐类别分成两组，每组 6 条。组名就是分类模型要学的**标签**。

In [1]:
features = ["工具", "杂志", "电影票", "游戏机"]

beer_rows = [
    [1, 0, 0, 1],
    [0, 0, 1, 0],
    [1, 1, 0, 1],
    [0, 0, 0, 1],
    [1, 1, 0, 1],
    [1, 0, 0, 1],
]

lipstick_rows = [
    [0, 1, 1, 0],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 1],
    [0, 0, 1, 0],
    [0, 1, 1, 0],
]

print("每行的列顺序", features)

print("推荐啤酒的记录")
for row in beer_rows:
    print(row)

print("推荐口红的记录")
for row in lipstick_rows:
    print(row)

每行的列顺序 ['工具', '杂志', '电影票', '游戏机']
推荐啤酒的记录
[1, 0, 0, 1]
[0, 0, 1, 0]
[1, 1, 0, 1]
[0, 0, 0, 1]
[1, 1, 0, 1]
[1, 0, 0, 1]
推荐口红的记录
[0, 1, 1, 0]
[1, 0, 1, 0]
[0, 1, 0, 1]
[0, 0, 1, 1]
[0, 0, 1, 0]
[0, 1, 1, 0]


### 2. 给同一位顾客手算两个类别的得分

只买了游戏机，对应 `[0, 0, 0, 1]`。先看「推荐啤酒的记录」，数一数其中有几条的第一位是 `0`，也就是没买工具。

In [2]:
customer = [0, 0, 0, 1]
print("顾客的购买记录", customer)

顾客的购买记录 [0, 0, 0, 1]


按顾客的四个观察值逐项计数。每项条件概率的分母是相应类别的记录总数。

| 手算项目 | 啤酒类 | 口红类 |
|---|---:|---:|
| 类别先验 | $6/12$ | $6/12$ |
| 没买工具 | $2/6$ | $5/6$ |
| 没买杂志 | $4/6$ | $3/6$ |
| 没买电影票 | $5/6$ | $1/6$ |
| 买了游戏机 | $5/6$ | $2/6$ |

在给定类别后，朴素贝叶斯将各项条件概率相乘，再乘类别先验。对这位顾客，两个类别分别得到

$$
\begin{aligned}
S(\text{啤酒}) &= \frac{6}{12}\cdot\frac{2}{6}\cdot\frac{4}{6}\cdot\frac{5}{6}\cdot\frac{5}{6} \approx 0.07716,\\
S(\text{口红}) &= \frac{6}{12}\cdot\frac{5}{6}\cdot\frac{3}{6}\cdot\frac{1}{6}\cdot\frac{2}{6} \approx 0.01157.
\end{aligned}
$$

啤酒的得分较高，所以推荐啤酒。

### 3. 用 Python 复现手算

**先算一项。** 一次看一条啤酒组记录。记录里的「工具」和顾客的「工具」都是第一位，两者相同就把计数加 1。

In [3]:
no_tool_count = 0

for row in beer_rows:
    if row[0] == customer[0]:  # [0] 取列表的第一项，也就是工具
        no_tool_count = no_tool_count + 1

print(f"啤酒组没买工具的条件概率 = {no_tool_count}/{len(beer_rows)}")

啤酒组没买工具的条件概率 = 2/6


**再算两个组的四列。** 列的位置从 `0` 开始。每换一列，就从 0 开始重新数两组里有多少条记录和顾客相同。例如「工具」要匹配 `0`，「游戏机」要匹配 `1`。数完一列，用 `append` 把结果加到各组的计数列表末尾。

In [4]:
columns = [0, 1, 2, 3]  # 工具、杂志、电影票、游戏机
beer_counts = []
lipstick_counts = []

for column in columns:
    beer_count = 0
    for row in beer_rows:
        if row[column] == customer[column]:
            beer_count = beer_count + 1
    beer_counts.append(beer_count)

    lipstick_count = 0
    for row in lipstick_rows:
        if row[column] == customer[column]:
            lipstick_count = lipstick_count + 1
    lipstick_counts.append(lipstick_count)

    print(features[column], "｜啤酒匹配", beer_count, "条｜口红匹配", lipstick_count, "条")

工具 ｜啤酒匹配 2 条｜口红匹配 5 条
杂志 ｜啤酒匹配 4 条｜口红匹配 3 条
电影票 ｜啤酒匹配 5 条｜口红匹配 1 条
游戏机 ｜啤酒匹配 5 条｜口红匹配 2 条


**再算得分。** 类别先验等于该组总数除以全部记录数。每个类别先把得分设为自己的先验，再依次乘上四项条件概率。每项条件概率都用「匹配条数 ÷ 该组总数」计算。

In [5]:
beer_total = len(beer_rows)
lipstick_total = len(lipstick_rows)
total = beer_total + lipstick_total

beer_prior = beer_total / total
lipstick_prior = lipstick_total / total

beer_score = beer_prior
lipstick_score = lipstick_prior

for column in columns:
    beer_probability = beer_counts[column] / beer_total
    lipstick_probability = lipstick_counts[column] / lipstick_total
    beer_score = beer_score * beer_probability
    lipstick_score = lipstick_score * lipstick_probability
    print(
        features[column],
        f"｜啤酒 {beer_counts[column]}/{beer_total}",
        f"｜口红 {lipstick_counts[column]}/{lipstick_total}",
    )

print("啤酒得分", round(beer_score, 5))
print("口红得分", round(lipstick_score, 5))

if beer_score >= lipstick_score:
    print("推荐商品：啤酒")
else:
    print("推荐商品：口红")

工具 ｜啤酒 2/6 ｜口红 5/6
杂志 ｜啤酒 4/6 ｜口红 3/6
电影票 ｜啤酒 5/6 ｜口红 1/6
游戏机 ｜啤酒 5/6 ｜口红 2/6
啤酒得分 0.07716
口红得分 0.01157
推荐商品：啤酒


### 4. 加一平滑

如果某个取值在某一类训练记录中出现了 0 次，条件概率直接按频率估计会得到 0，连乘后该类得分也会变成 0。加一平滑给「买了」和「没买」各加一次计数。每个特征有两个取值，所以分母加 2。

用啤酒类的「没买工具」举例，$2/6$ 平滑后变为 $(2+1)/(6+2)=3/8$。先验仍按实际类别比例计算。

沿用刚才的两个计数列表，用平滑后的条件概率重算得分，再让每个类别的得分除以两个得分之和，得到这位顾客的**类别概率**。

In [6]:
beer_smoothed_score = beer_prior
lipstick_smoothed_score = lipstick_prior

for column in columns:
    beer_probability = (beer_counts[column] + 1) / (beer_total + 2)
    lipstick_probability = (lipstick_counts[column] + 1) / (lipstick_total + 2)
    beer_smoothed_score = beer_smoothed_score * beer_probability
    lipstick_smoothed_score = lipstick_smoothed_score * lipstick_probability

total_score = beer_smoothed_score + lipstick_smoothed_score
manual_beer_probability = beer_smoothed_score / total_score
manual_lipstick_probability = lipstick_smoothed_score / total_score

print("啤酒概率", round(manual_beer_probability, 6))
print("口红概率", round(manual_lipstick_probability, 6))

啤酒概率 0.789474
口红概率 0.210526


## 第二部分｜用 scikit-learn 建模和预测

### 1. 准备训练数据

把前面的两组记录接成一个列表 `X_train`，啤酒组在前，口红组在后。按相同顺序给每条记录配一个类别标签，组成 `y_train`。

In [32]:
X_train = beer_rows + lipstick_rows
y_train = ["啤酒"] * len(beer_rows) + ["口红"] * len(lipstick_rows)

print("训练记录数:", len(X_train))
print("第一条记录:", X_train[0], "对应的标签:", y_train[0])

训练记录数: 12
第一条记录: [1, 0, 0, 1] 对应的标签: 啤酒


### 2. 训练模型

购买记录的每一列都是 0 或 1，所以用 `BernoulliNB`。设置 `alpha=1`，让模型使用第一部分的加一平滑。调用 `fit`，把购买记录和对应的标签交给模型学习。

In [ ]:
from sklearn.naive_bayes import BernoulliNB

model = BernoulliNB(alpha=1)
model.fit(X_train, y_train)

model_classes = list(model.classes_)
print("模型学到的类别", model_classes)

模型学到的类别 [np.str_('口红'), np.str_('啤酒')]


### 3. 预测推荐商品和类别概率

把「只买了游戏机」的顾客交给模型。`[customer]` 是一个只含一条记录的列表。`predict` 给出推荐商品，`predict_proba` 给出两类概率。概率的排列顺序对应 `model.classes_`，先找到两个类别各自的位置，再取出对应的概率。

In [9]:
prediction = model.predict([customer])[0]
model_probabilities = model.predict_proba([customer])[0]

beer_index = model_classes.index("啤酒")
lipstick_index = model_classes.index("口红")
model_beer_probability = model_probabilities[beer_index]
model_lipstick_probability = model_probabilities[lipstick_index]

print("推荐商品", prediction)
print("啤酒概率", round(model_beer_probability, 6))
print("口红概率", round(model_lipstick_probability, 6))

推荐商品 啤酒
啤酒概率 0.789474
口红概率 0.210526


### 4. 核对两种方法的结果

对同一位顾客，两种方法都用了同一份训练数据和加一平滑。把第一部分用 Python 算出的类别概率与模型预测的概率逐项比较。

In [10]:
print("啤酒｜自己算", round(manual_beer_probability, 6), "｜模型", round(model_beer_probability, 6))
print("口红｜自己算", round(manual_lipstick_probability, 6), "｜模型", round(model_lipstick_probability, 6))

啤酒｜自己算 0.789474 ｜模型 0.789474
口红｜自己算 0.210526 ｜模型 0.210526


### 5. 动手练习｜如果顾客连游戏机也没买

将最后一位从 `1` 改成 `0`，这位顾客就成了四样商品都没买的人。运行代码前，先判断两类概率会往哪个方向变。运行后，用两类记录里买过游戏机的人数解释结果。

In [ ]:
practice_customer = [0, 0, 0, 1]  # 请把最后一位改成 0

practice_prediction = model.predict([practice_customer])[0]
practice_probabilities = model.predict_proba([practice_customer])[0]

print("推荐商品:", practice_prediction)
print("啤酒概率:", round(practice_probabilities[beer_index], 6))
print("口红概率:", round(practice_probabilities[lipstick_index], 6))

推荐商品 啤酒
啤酒概率 0.789474
口红概率 0.210526


**课堂记录**

- 我改动了哪个特征？
- 推荐结果怎样变化？
- 两类训练记录中，这项特征的计数如何解释概率的变化？

在这里记录你的回答。